# Nemotron v1 SFT — Colab Pro+ A100

**Open in Colab Pro+, pick A100 GPU + High-RAM, then Run All.**

Repo: https://github.com/ykaya-jp/NVIDIA-Nemotron-Model-Reasoning-Challenge

**Hot-reload workflow**: I push a fix → user re-runs cells 1, 4, 5 (cells 2/3 stay live).

Nemotron v1 SFT — Colab Pro+ A100 version.

> USAGE (Colab Pro+):
> 1. New Notebook → Runtime → Change runtime type → A100 (40 GB GPU,
>    High-RAM). Save & connect.
> 2. Paste this whole script into one cell.
> 3. Run All. Expected runtime: 6-10 h on A100 40 GB.
> 4. The trained LoRA adapter lands at
>    /content/drive/MyDrive/nemotron-2026/adapter_v1/. After it
>    finishes, download that folder (or upload it as a Kaggle Dataset
>    `ky7240/nemotron-v1-adapter`) and reference it from a thin
>    submission notebook (= GPU off, just writes submission.csv with
>    the adapter path) to submit to Kaggle.

ATTRIBUTION (= Apache 2.0 licensed public kernels we draw setup code
from; please credit when publishing):
- dgxchen/training-with-unsloth-to-achieve-0-85-lb (Kaggle kernel)
- konbu17/nemotron-sft-lora-with-cot (Kaggle kernel)
- huikang/end-to-end-finetuning-for-lb-0-85 (Kaggle kernel)

NOVEL CONTRIBUTION = the SFT data (data/processed/train_sft_verifier_
only.jsonl, 5418 records). It is generated by 5 deterministic Python
solvers in this repo (Roman / Physics / Unit / Cipher / Bit) producing
verifier-backed Chain-of-Thought traces. See
docs/strategy/winning-strategy.dense.md for the full rationale.

## Cell 1 — Drive mount + git clone-or-pull (hot-reload safe)

In [1]:
# Designed so re-running this cell alone refreshes the repo to the
# latest origin/main without touching cells 2 (pip install) or 3
# (model load). Cells 4 and 5 are thin wrappers around
# `colab_pipeline.py` in the repo, so any fix lands in the running
# kernel by simply re-running cells 1 -> 4 -> 5.
from google.colab import drive

drive.mount("/content/drive")

import os
import subprocess

REPO_URL = "https://github.com/ykaya-jp/NVIDIA-Nemotron-Model-Reasoning-Challenge.git"
REPO_DIR = "/content/nemotron"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    # Already cloned — fast-forward to the latest commit on origin/main.
    # `reset --hard` is safe because we never edit files inside the
    # Colab clone (the user edits happen on the host machine and reach
    # us through git).
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "--depth", "1",
                    "origin", "main"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard",
                    "origin/main"], check=True)
os.chdir(REPO_DIR)
print("repo at", REPO_DIR)
subprocess.run(["git", "log", "-1", "--oneline"])

ADAPTER_DIR = "/content/drive/MyDrive/nemotron-2026/adapter_v1"
os.makedirs(ADAPTER_DIR, exist_ok=True)

# Make the repo's Python package importable. Adding sys.path here
# (rather than in cell 4) means an `importlib.reload(...)` in cell 4
# always sees the file freshly written by the git pull above.
import sys
if os.path.join(REPO_DIR, "src") not in sys.path:
    sys.path.insert(0, os.path.join(REPO_DIR, "src"))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
repo at /content/nemotron


## Cell 2 — Dependencies (plain transformers + peft + TRL stack)

In [2]:
# Why no Unsloth: Unsloth issue #3480 (= "Cannot load Nvidia Nemotron
# Nano models") + a meta-tensor crash inside unsloth_zoo
# fix_untrained_tokens make the Unsloth fast-path unusable on this
# specific model. The NVIDIA / DataCamp reference notebook uses plain
# transformers + peft + TRL with the version pins below; this is the
# documented working stack for Nemotron-3-Nano-30B-A3B-BF16 on an
# A100 80 GB.
# References:
# - https://www.datacamp.com/tutorial/fine-tuning-nvidia-nemotron-3-nano
# - https://huggingface.co/nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16
# - https://github.com/unslothai/unsloth/issues/3480
import subprocess
import sys


def _run(cmd):
    """Run shell command, stream output to the cell so the user sees errors."""
    print(f"$ {' '.join(cmd)}")
    subprocess.run(cmd, check=True)


# Step 1: build tooling for the Mamba CUDA extensions.
_run([sys.executable, "-m", "pip", "install", "-q", "-U", "packaging", "ninja"])

# Step 2: pin torch to the CUDA 12.8 build required by Nemotron-3-Nano.
# Colab default torch is usually fine, but the cu128 wheel is the one
# NVIDIA / Unsloth document for the Mamba kernels, so we force it.
_run([sys.executable, "-m", "pip", "uninstall", "-y",
      "torch", "torchvision", "torchaudio", "triton"])
_run([sys.executable, "-m", "pip", "install", "-q",
      "torch==2.7.1", "torchvision==0.22.1", "torchaudio==2.7.1",
      "--index-url", "https://download.pytorch.org/whl/cu128"])

# Step 3: HF stack with the exact pins from the NVIDIA / DataCamp
# reference. transformers 4.56.2 + trl 0.22.2 supports
# `completion_only_loss=True` and `processing_class=tokenizer`.
# peft is pinned to <0.16 on purpose: peft 0.16+ requires
# torchao>=0.16, which in turn needs torch>=2.8 (incompatible with
# the torch 2.7.1 + CUDA 12.8 build we pinned for the Mamba kernels).
# We don't use any torchao features (no quantization), so this is the
# cleanest resolution.
_run([sys.executable, "-m", "pip", "install", "-q", "-U",
      "transformers==4.56.2", "tokenizers", "trl==0.22.2",
      "accelerate", "datasets", "peft<0.16",
      "huggingface_hub", "safetensors",
      "sentencepiece", "nbformat"])

# Step 3b: defence-in-depth — even with peft<0.16 pinned, a stale
# torchao 0.10 may live in the Colab base image and trip
# `peft.import_utils.is_torchao_available()` (it import-checks the
# minimum version *if* torchao is present). Uninstalling torchao
# makes that helper return False without raising. We never use
# torchao quantization, so removing it is safe.
_run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"])

# Step 4: Mamba-SSM + causal-conv1d at the versions that ship in the
# NVIDIA reference. `--no-build-isolation` reuses the torch we just
# installed (otherwise pip spawns a clean env without torch and the
# CUDA build fails). First build ~5-10 min on A100; cached after.
_run([sys.executable, "-m", "pip", "install", "-q",
      "-U", "--no-build-isolation",
      "mamba_ssm==2.2.5", "causal_conv1d==1.5.2"])

import torch

print(
    "cuda:",
    torch.cuda.is_available(),
    "device:",
    torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
    "mem GB:",
    torch.cuda.get_device_properties(0).total_memory / 1024**3 if torch.cuda.is_available() else 0,
)


$ /usr/bin/python3 -m pip install -q -U packaging ninja
$ /usr/bin/python3 -m pip uninstall -y torch torchvision torchaudio triton
$ /usr/bin/python3 -m pip install -q torch==2.7.1 torchvision==0.22.1 torchaudio==2.7.1 --index-url https://download.pytorch.org/whl/cu128
$ /usr/bin/python3 -m pip install -q -U transformers==4.56.2 tokenizers trl==0.22.2 accelerate datasets peft<0.16 huggingface_hub safetensors sentencepiece nbformat
$ /usr/bin/python3 -m pip uninstall -y torchao
$ /usr/bin/python3 -m pip install -q -U --no-build-isolation mamba_ssm==2.2.5 causal_conv1d==1.5.2
cuda: True device: NVIDIA A100-SXM4-80GB mem GB: 79.250732421875


## Cell 3 — Load base model + tokenizer (plain transformers path)

In [3]:
import transformers.tokenization_utils_base
import transformers.utils.hub
from huggingface_hub.utils import EntryNotFoundError

_orig_list_repo_templates = transformers.utils.hub.list_repo_templates


def _safe_list_repo_templates(*args, **kwargs):
    try:
        return _orig_list_repo_templates(*args, **kwargs)
    except EntryNotFoundError:
        return []
    except Exception as e:
        msg = str(e)
        if "404" in msg or "Entry Not Found" in msg or "does not exist" in msg:
            return []
        raise


transformers.utils.hub.list_repo_templates = _safe_list_repo_templates
if hasattr(transformers.tokenization_utils_base, "list_repo_templates"):
    transformers.tokenization_utils_base.list_repo_templates = _safe_list_repo_templates
print("✓ patched list_repo_templates")

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MAX_SEQ_LEN = 2048
MODEL_NAME = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

if "model" in globals():
    try:
        del model  # noqa: F821
    except Exception:
        pass
import gc
gc.collect()
torch.cuda.empty_cache()

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    dtype=torch.bfloat16,
    device_map={"": 0},
    attn_implementation="eager",
)
model.config.use_cache = False
model.config.pad_token_id = tokenizer.pad_token_id
model.config.eos_token_id = tokenizer.eos_token_id

meta_params = [n for n, p in model.named_parameters() if p.is_meta]
assert not meta_params, f"still meta: {meta_params[:5]}"
print("✓ base model loaded on GPU 0; no meta tensors remain")

✓ patched list_repo_templates to tolerate missing additional_chat_templates/


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/13 [00:00<?, ?it/s]

✓ base model loaded on GPU 0; no meta tensors remain


## Cell 4 — Load SFT data + chat template + upsample + length audit

In [4]:
import importlib
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/ykaya-jp/NVIDIA-Nemotron-Model-Reasoning-Challenge.git"
REPO_DIR = "/content/nemotron"

_missing = [v for v in ("tokenizer", "model", "MAX_SEQ_LEN") if v not in globals()]
if _missing:
    raise RuntimeError(f"Cell 4 prerequisites missing: {_missing}. Re-run Cell 3 first.")


def _try(cmd):
    print("$", " ".join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.stdout:
        print(r.stdout, end="")
    if r.returncode != 0 and r.stderr:
        print("stderr:", r.stderr, end="")
    return r.returncode


git_dir = os.path.join(REPO_DIR, ".git")
rc = 128
if os.path.isdir(git_dir):
    rc = _try(["git", "-C", REPO_DIR, "fetch", "--depth", "20", "origin", "main"])
    if rc != 0:
        rc = _try(["git", "-C", REPO_DIR, "fetch", "--unshallow", "origin", "main"])
    if rc != 0:
        rc = _try(["git", "-C", REPO_DIR, "fetch", "origin", "main"])

if rc != 0 or not os.path.isdir(git_dir):
    print("→ falling back to a fresh clone")
    if os.path.isdir(REPO_DIR):
        shutil.rmtree(REPO_DIR)
    _try(["git", "clone", REPO_URL, REPO_DIR])

_try(["git", "-C", REPO_DIR, "reset", "--hard", "origin/main"])
_try(["git", "-C", REPO_DIR, "log", "-1", "--oneline"])

src_path = os.path.join(REPO_DIR, "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

mod_name = "nvidia_nemotron_model_reasoning_challenge.colab_pipeline"
if mod_name in sys.modules:
    del sys.modules[mod_name]
import nvidia_nemotron_model_reasoning_challenge.colab_pipeline as _pipeline
importlib.reload(_pipeline)

SFT_PATH = os.path.join(REPO_DIR, "data/processed/train_sft_verifier_only.jsonl")
rows = _pipeline.load_rows(SFT_PATH)
ds = _pipeline.build_dataset(rows, tokenizer, MAX_SEQ_LEN)

$ git -C /content/nemotron fetch --depth 20 origin main
$ git -C /content/nemotron reset --hard origin/main
HEAD is now at 4c143b4 fix(colab): force device_map={"": 0} to keep MoE experts off CPU/disk
$ git -C /content/nemotron log -1 --oneline
4c143b4 fix(colab): force device_map={"": 0} to keep MoE experts off CPU/disk
loaded 5418 verifier-backed records
source distribution: Counter({'solver_physics': 1597, 'solver_unit': 1594, 'solver_roman': 1576, 'solver_cipher': 605, 'solver_bit': 46})
after upsampling: 6575 records
length audit on 200-sample: max=708 p95=670 truncated@2048=0 (0.0%)


## Cell 5 — SFT training (1 epoch, completion-only loss, LoRA via TRL)

In [ ]:
import importlib
import inspect
import os
import sys

mod_name = "nvidia_nemotron_model_reasoning_challenge.colab_pipeline"
if mod_name in sys.modules:
    del sys.modules[mod_name]
import nvidia_nemotron_model_reasoning_challenge.colab_pipeline as _pipeline
importlib.reload(_pipeline)

src = inspect.getsource(_pipeline.train_lora)
assert "save_steps=50" in src or "save_steps = 50" in src, (
    "colab_pipeline did NOT pick up the new save_steps=50. "
    "Check `git log -1` above — HEAD should be b4d554c or later."
)
assert "resume_from_checkpoint" in src, (
    "colab_pipeline did NOT pick up resume_from_checkpoint. "
    "Re-run Cell 4 to force a fresh git pull."
)
print("✓ colab_pipeline reloaded: save_steps=50, Drive checkpoint, auto-resume")

OUTPUT_DIR = "/content/drive/MyDrive/nemotron-2026/checkpoints_v1"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Show whether we'll resume or start fresh.
existing = [d for d in os.listdir(OUTPUT_DIR) if d.startswith("checkpoint-")]
if existing:
    last = max(existing, key=lambda d: int(d.split("-")[1]))
    print(f"   found prior checkpoint(s): {existing} → resuming from {last}")
else:
    print(f"   no prior checkpoint at {OUTPUT_DIR}; starting fresh")

trainer = _pipeline.train_lora(
    model, tokenizer, ds,
    output_dir=OUTPUT_DIR,
    max_seq_len=MAX_SEQ_LEN,
)

# Save the final adapter into a sibling Drive directory so the
# checkpoints_v1/ folder stays as resume cache.
ADAPTER_DIR = "/content/drive/MyDrive/nemotron-2026/adapter_v1"
os.makedirs(ADAPTER_DIR, exist_ok=True)
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"saved final adapter to {ADAPTER_DIR}")
for fn in sorted(os.listdir(ADAPTER_DIR)):
    sz = os.path.getsize(os.path.join(ADAPTER_DIR, fn))
    print(f"  {fn}: {sz:,} bytes")


✓ colab_pipeline reloaded with torchao bypass + MoE-friendly device_map


Tokenizing train dataset:   0%|          | 0/6575 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/6575 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 11}.


Step,Training Loss
20,17.451700
40,1.376600
60,1.056500
80,1.012000
100,0.994800
120,0.912700
140,0.777600
160,0.849700


## Cell 6 — Save adapter to Drive

In [1]:
# After training the model object is a PeftModel wrapping the base.
# save_pretrained writes ONLY the LoRA adapter (~200-400 MB), which is
# what the Kaggle metric notebook expects.
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("saved adapter to", ADAPTER_DIR)
for fn in sorted(os.listdir(ADAPTER_DIR)):
    sz = os.path.getsize(os.path.join(ADAPTER_DIR, fn))
    print(f"  {fn}: {sz:,} bytes")
print()
print("Next: upload this folder to Kaggle as `ky7240/nemotron-v1-adapter`,")
print("then push the thin submission notebook (= GPU off) to Kaggle and submit.")


NameError: name 'trainer' is not defined